# AI Movie Ticket Booking Agent

**Hugging Face + LangChain + LangGraph | Google Colab**

This is a simple movie-booking assistant. You can chat with it to select a **city, movie, showtime, and number of seats**. It remembers your choices and gives a demo booking confirmation.

**How it works:** Hugging Face understands your message → LangGraph remembers your choices → LangChain tools check shows and create a demo booking.


## 1. Install the libraries

In [1]:
!pip -q install -U transformers accelerate langchain-core langchain-huggingface langgraph

## 2. Load the Hugging Face model

We use a small Qwen model. The first time you run this cell, Colab downloads the model. Public model access does not require an API key.

In [2]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Hugging Face model runs in Colab, not through a paid API.
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto",
    dtype="auto",
    max_new_tokens=120,
    do_sample=False,
    return_full_text=False,
)

hf_llm = HuggingFacePipeline(pipeline=generator)
llm = ChatHuggingFace(llm=hf_llm)
print("Hugging Face model loaded!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Hugging Face model loaded!


## 3. Create a small movie database

These movies, times, prices, and seats are **fictional examples** for learning and demonstrations.

In [4]:
SHOWS = {
    "Delhi": {
        "Interstellar": {"6:00 PM": {"price": 250, "seats": 12}, "9:00 PM": {"price": 300, "seats": 10}},
        "Inception": {"5:00 PM": {"price": 220, "seats": 15}, "8:00 PM": {"price": 260, "seats": 9}},
    },
    "Mumbai": {
        "Interstellar": {"4:00 PM": {"price": 280, "seats": 10}, "7:00 PM": {"price": 320, "seats": 8}},
        "Inception": {"6:00 PM": {"price": 240, "seats": 14}, "9:00 PM": {"price": 290, "seats": 7}},
    },
    "Pune": {
        "Interstellar": {"5:30 PM": {"price": 200, "seats": 11}, "8:30 PM": {"price": 240, "seats": 9}},
        "Inception": {"4:30 PM": {"price": 210, "seats": 10}, "7:30 PM": {"price": 250, "seats": 12}},
    },
}

BOOKINGS = {}  # Confirmed demo bookings are stored here during this Colab session.
print("Available cities:", ", ".join(SHOWS))

Available cities: Delhi, Mumbai, Pune


## 4. Build three simple LangChain tools

The agent uses these tools to show movies, find showtimes, and confirm a simulated booking.

In [5]:
from langchain_core.tools import tool
import uuid

@tool
def list_movies(city: str) -> str:
    """List the demo movies available in a city."""
    return ", ".join(SHOWS.get(city, {})) or "No demo shows in this city."

@tool
def list_showtimes(city: str, movie: str) -> str:
    """List showtimes, ticket prices, and remaining seats for a movie."""
    shows = SHOWS.get(city, {}).get(movie, {})
    if not shows:
        return "No demo shows found."
    return ", ".join(
        f"{time} (₹{info['price']}, {info['seats']} seats left)"
        for time, info in shows.items()
    )

@tool
def book_tickets(city: str, movie: str, showtime: str, seats: int) -> str:
    """Confirm a simulated booking and reduce the demo seats remaining."""
    show = SHOWS.get(city, {}).get(movie, {}).get(showtime)
    if not show:
        return "FAILED: That show is not available."
    if not 1 <= seats <= 6:
        return "FAILED: Choose 1 to 6 seats."
    if show["seats"] < seats:
        return f"FAILED: Only {show['seats']} seats remain."

    show["seats"] -= seats
    booking_id = "DEMO-" + uuid.uuid4().hex[:8].upper()
    total = seats * show["price"]
    BOOKINGS[booking_id] = {
        "city": city, "movie": movie, "showtime": showtime,
        "seats": seats, "total": total,
    }
    return f"CONFIRMED: {booking_id} | {movie} | {city} | {showtime} | {seats} seats | ₹{total}"

print("Tools ready: list_movies, list_showtimes, book_tickets")

Tools ready: list_movies, list_showtimes, book_tickets


## 5. Let the model understand the user's message

LangChain asks Qwen to identify any booking details the user mentioned. Missing details stay empty, so the assistant can ask about them one at a time.

In [6]:
import json
import re
from datetime import datetime
from typing import TypedDict

# We keep a small state so the agent remembers previous selections.
class BookingState(TypedDict, total=False):
    message: str
    city: str
    movie: str
    showtime: str
    seats: int
    confirm: bool
    reset: bool
    review_ready: bool
    confirmed: bool
    reply: str

prompt = ChatPromptTemplate.from_messages([
    ("system", """Extract booking details ONLY from the user's latest message.
Cities: Delhi, Mumbai, Pune. Movies: Interstellar, Inception.
Return only one JSON object with these keys:
{{"city":"", "movie":"", "showtime":"", "seats":0}}
Use empty strings or 0 for details not mentioned. Do not guess.
Examples:
'I want two seats for Interstellar' -> {{"city":"", "movie":"Interstellar", "showtime":"", "seats":2}}
'Mumbai at 7 pm' -> {{"city":"Mumbai", "movie":"", "showtime":"7:00 PM", "seats":0}}"""),
    ("human", "{message}"),
])
extract_chain = prompt | llm | StrOutputParser()


def normalize_time(value):
    value = str(value).strip().upper().replace(".", "")
    for fmt in ("%I:%M %p", "%I %p", "%I%p", "%H:%M"):
        try:
            return datetime.strptime(value, fmt).strftime("%-I:%M %p")
        except ValueError:
            pass
    return value


def get_details(message):
    # The model may occasionally add extra text, so read just the JSON part.
    result = extract_chain.invoke({"message": message})
    match = re.search(r"\{[^{}]*\}", result, flags=re.DOTALL)
    try:
        details = json.loads(match.group()) if match else {}
    except json.JSONDecodeError:
        details = {}

    # Simple checks help if the small model misses a city or movie name.
    for city in SHOWS:
        if city.lower() in message.lower():
            details["city"] = city
    for movie in {m for city in SHOWS.values() for m in city}:
        if movie.lower() in message.lower():
            details["movie"] = movie

    # Recognize times such as "6 pm" or "6:00 PM", even if the model misses them.
    time_match = re.search(r"\b(1[0-2]|0?[1-9])(?::([0-5]\d))?\s*(am|pm)\b", message, re.I)
    if time_match:
        hour = int(time_match.group(1))
        minute = int(time_match.group(2) or 0)
        details["showtime"] = f"{hour}:{minute:02d} {time_match.group(3).upper()}"

    seat_match = re.search(r"\b([1-6])\s*(?:seats?|tickets?)\b", message, re.I)
    if seat_match:
        details["seats"] = int(seat_match.group(1))
    if message.strip().isdigit():
        details["seats"] = int(message.strip())
    for word, number in {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6}.items():
        if re.search(rf"\b{word}\s+(?:seats?|tickets?)\b", message, re.I):
            details["seats"] = number

    return details

## 6. Build the LangGraph workflow

Two steps: **understand the message** and **decide what to do next**. The graph remembers your choices between messages.

In [11]:
def understand(state: BookingState):
    text = state["message"].strip()
    simple = text.lower()

    # Confirm booking only when the user explicitly agrees
    if simple in {"confirm", "yes", "yes please", "book it", "go ahead"}:
        return {"confirm": True, "reset": False}

    # Start a new booking
    if simple in {"new booking", "restart", "reset", "start over"}:
        return {
            "city": "", "movie": "", "showtime": "", "seats": 0,
            "confirm": False, "reset": True, "review_ready": False,
            "confirmed": False,
        }

    details = get_details(text)

    # FIX 1: Don't change the movie unless the user mentions it
    if not any(movie.lower() in simple
               for movies in SHOWS.values()
               for movie in movies):
        details["movie"] = ""

    # FIX 2: Don't change the city unless the user mentions it
    if not any(city.lower() in simple for city in SHOWS):
        details["city"] = ""

    changes = {"confirm": False, "reset": False}

    city = str(details.get("city") or "").strip()
    movie = str(details.get("movie") or "").strip()
    showtime = str(details.get("showtime") or "").strip()

    # Update city
    if city.lower() in {c.lower() for c in SHOWS}:
        city = next(c for c in SHOWS if c.lower() == city.lower())

        if city != state.get("city"):
            changes.update({
                "city": city,
                "movie": "",
                "showtime": ""
            })

    # Update movie
    all_movies = {m for city_shows in SHOWS.values() for m in city_shows}

    if movie.lower() in {m.lower() for m in all_movies}:
        movie = next(m for m in all_movies if m.lower() == movie.lower())

        if movie != state.get("movie"):
            changes.update({
                "movie": movie,
                "showtime": ""
            })

    # Update showtime
    if showtime:
        changes["showtime"] = normalize_time(showtime)

    # Update number of seats
    try:
        seats = int(details.get("seats") or 0)
    except (TypeError, ValueError):
        seats = 0

    if seats > 0:
        changes["seats"] = seats

    # Booking must be reviewed if details change
    if any(k in changes for k in ("city", "movie", "showtime", "seats")):
        changes["review_ready"] = False

    return changes

In [12]:
def respond(state: BookingState):
    if state.get("reset"):
        return {"reply": "Sure! Let's start a new booking. Choose a city: Delhi, Mumbai, or Pune."}

    if state.get("confirmed"):
        return {"reply": "Your demo booking is already confirmed. Type 'new booking' to start again."}

    city = state.get("city", "")
    movie = state.get("movie", "")
    showtime = state.get("showtime", "")
    seats = state.get("seats", 0)

    if not city:
        return {"reply": "Which city would you like? Delhi, Mumbai, or Pune?"}
    if not movie:
        movies = list_movies.invoke({"city": city})
        return {"reply": f"In {city}, you can watch: {movies}. Which movie would you like?"}
    if movie not in SHOWS.get(city, {}):
        return {"reply": f"{movie} isn't in our {city} demo listings. Choose: {list_movies.invoke({'city': city})}."}
    if not showtime:
        times = list_showtimes.invoke({"city": city, "movie": movie})
        return {"reply": f"Available shows for {movie} in {city}: {times}. Which time works for you?"}
    show = SHOWS[city][movie].get(showtime)
    if show is None:
        times = list_showtimes.invoke({"city": city, "movie": movie})
        return {"reply": f"That showtime isn't available. Please choose from: {times}."}
    if not 1 <= seats <= 6:
        return {"reply": "How many seats would you like? Please choose 1 to 6."}
    if show["seats"] < seats:
        return {"review_ready": False,
                "reply": f"Only {show['seats']} demo seats are left for this show. Please choose fewer seats."}

    total = seats * show["price"]
    if state.get("confirm") and state.get("review_ready"):
        result = book_tickets.invoke({
            "city": city, "movie": movie, "showtime": showtime, "seats": seats
        })
        if result.startswith("CONFIRMED:"):
            return {"confirmed": True,
                    "reply": f"🎟️ Booking confirmed (demo only)!\n{result}\nNo real tickets were purchased."}
        return {"review_ready": False, "reply": result}

    return {
        "review_ready": True,
        "reply": (f"Here's your booking summary:\n"
                  f"Movie: {movie}\nCity: {city}\nShowtime: {showtime}\n"
                  f"Seats: {seats}\nTotal: ₹{total}\n\n"
                  "Type 'confirm' to make a DEMO booking, or change any detail."),
    }

In [13]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

builder = StateGraph(BookingState)
builder.add_node("understand", understand)
builder.add_node("respond", respond)
builder.add_edge(START, "understand")
builder.add_edge("understand", "respond")
builder.add_edge("respond", END)

# Memory keeps selections between messages for the same conversation ID.
agent = builder.compile(checkpointer=InMemorySaver())
print("LangGraph ready: START → understand → respond → END")

LangGraph ready: START → understand → respond → END


## 7. Try a complete demo booking

This example uses the sample Delhi shows. You can watch how the assistant remembers each answer.

In [14]:
session = {
    "configurable": {
        "thread_id": "movie-demo-" + uuid.uuid4().hex[:8]
    }
}

def chat(message):
    result = agent.invoke({"message": message}, config=session)
    print("You:", message)
    print("Agent:", result["reply"])
    print("-" * 45)
    return result

chat("I want to watch a movie in Delhi")
chat("Interstellar")
chat("6:00 PM, 2 seats")
chat("confirm")

You: I want to watch a movie in Delhi
Agent: In Delhi, you can watch: Interstellar, Inception. Which movie would you like?
---------------------------------------------
You: Interstellar
Agent: Available shows for Interstellar in Delhi: 6:00 PM (₹250, 12 seats left), 9:00 PM (₹300, 10 seats left). Which time works for you?
---------------------------------------------
You: 6:00 PM, 2 seats
Agent: Here's your booking summary:
Movie: Interstellar
City: Delhi
Showtime: 6:00 PM
Seats: 2
Total: ₹500

Type 'confirm' to make a DEMO booking, or change any detail.
---------------------------------------------
You: confirm
Agent: 🎟️ Booking confirmed (demo only)!
CONFIRMED: DEMO-D113C9DB | Interstellar | Delhi | 6:00 PM | 2 seats | ₹500
No real tickets were purchased.
---------------------------------------------


{'message': 'confirm',
 'city': 'Delhi',
 'movie': 'Interstellar',
 'showtime': '6:00 PM',
 'seats': 2,
 'confirm': True,
 'reset': False,
 'review_ready': True,
 'confirmed': True,
 'reply': '🎟️ Booking confirmed (demo only)!\nCONFIRMED: DEMO-D113C9DB | Interstellar | Delhi | 6:00 PM | 2 seats | ₹500\nNo real tickets were purchased.'}

## 8. Chat with your agent

Run this cell to enter your own messages. Use **new booking** to start again, or **exit** to stop.

In [15]:
session = {
    "configurable": {
        "thread_id": "my-movie-chat-" + uuid.uuid4().hex[:8]
    }
}
print("Welcome! Ask me to book a movie. Type 'exit' to stop.")

while True:
    message = input("You: ").strip()
    if message.lower() in {"exit", "quit", "stop"}:
        print("Agent: Thanks for trying the demo!")
        break
    if message:
        answer = agent.invoke({"message": message}, config=session)
        print("Agent:", answer["reply"], "\n")

Welcome! Ask me to book a movie. Type 'exit' to stop.
You: i want to book ticket
Agent: Which city would you like? Delhi, Mumbai, or Pune? 

You: Pune
Agent: In Pune, you can watch: Interstellar, Inception. Which movie would you like? 

You: Inception
Agent: Available shows for Inception in Pune: 4:30 PM (₹210, 10 seats left), 7:30 PM (₹250, 12 seats left). Which time works for you? 

You: 4:30 PM
Agent: How many seats would you like? Please choose 1 to 6. 

You: 2
Agent: Here's your booking summary:
Movie: Inception
City: Pune
Showtime: 4:30 PM
Seats: 2
Total: ₹420

Type 'confirm' to make a DEMO booking, or change any detail. 

You: confirm
Agent: 🎟️ Booking confirmed (demo only)!
CONFIRMED: DEMO-FE850385 | Inception | Pune | 4:30 PM | 2 seats | ₹420
No real tickets were purchased. 

You: quit
Agent: Thanks for trying the demo!
